In [3]:
!pip install pyngrok
from pyngrok import ngrok
ngrok.kill()

In [1]:
# 1. Install dependencies
!pip install fastapi uvicorn pyngrok -q
!pip install diffusers torch imageio numpy nest_asyncio accelerate transformers

# 2. Write your FastAPI app
app_code = """
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import FileResponse

import torch
from diffusers import DiffusionPipeline
from diffusers.utils import export_to_video

app = FastAPI()

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

@app.get("/generate")
def generate(prompt: str):
    pipe = DiffusionPipeline.from_pretrained("damo-vilab/text-to-video-ms-1.7b", torch_dtype=torch.float16)
    pipe.enable_sequential_cpu_offload()

    video_frames = pipe(prompt=prompt, num_frames=80).frames[0]

    output_path = "output.mp4"
    export_to_video(video_frames, output_path)

    # Send the video file back as a downloadable response
    return FileResponse(
        path=output_path,
        media_type="video/mp4",
        filename="generated_video.mp4"
    )
"""

with open("main.py", "w") as f:
    f.write(app_code)

# 3. Start ngrok tunnel + FastAPI
from pyngrok import ngrok
import uvicorn, threading

ngrok.set_auth_token("3C1Isuoamlc7CmG9Ze6iHezapdA_32tcL7hCFfQGAtkb1Qerb")

public_url = ngrok.connect(8000)
print(f"✅ Your API URL: {public_url}")

def run():
    uvicorn.run("main:app", host="0.0.0.0", port=8000)

threading.Thread(target=run, daemon=True).start()

✅ Your API URL: NgrokTunnel: "https://semialcoholic-globularly-devona.ngrok-free.dev" -> "http://localhost:8000"
